In [1]:
import sqlite3
import pandas as pd
import re

In [2]:
conn = sqlite3.connect(r'C:\Users\Admin\Desktop\DA Project\FMCG\FMCG database.db')

In [3]:
pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table' OR type='view';", conn)

,name
0,FactSales
1,Customer_Type
2,Product_Type
3,Promotion_Type
4,Region
5,Sales_Channel
6,Sales_Person
7,Fact_Sales_Merged


In [4]:
df = pd.read_sql_query("SELECT * FROM FactSales", conn)
customer_type = pd.read_sql_query("SELECT * FROM Customer_Type", conn)
product_type = pd.read_sql_query("SELECT * FROM Product_Type", conn)
promotion_type = pd.read_sql_query("SELECT * FROM Promotion_Type", conn)
region = pd.read_sql_query("SELECT * FROM Region", conn)
sales_channel = pd.read_sql_query("SELECT * FROM Sales_Channel", conn)
sales_person = pd.read_sql_query("SELECT * FROM Sales_Person", conn)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20745 entries, 0 to 20744
Data columns (total 18 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Order_ID             20745 non-null  str    
 1   Order_Date           20150 non-null  str    
 2   Geo_ID               20337 non-null  float64
 3   SKU                  20434 non-null  str    
 4   SalesPerson_ID       20434 non-null  float64
 5   CustomerType_ID      20330 non-null  float64
 6   Channel_ID           20435 non-null  float64
 7   Promotion_ID         20130 non-null  float64
 8   Units_Sold           20745 non-null  float64
 9   Unit_Price_USD       20745 non-null  float64
 10  Discount_Pct         20745 non-null  float64
 11  Gross_Sales_USD      20745 non-null  float64
 12  Marketing_Spend_USD  20745 non-null  float64
 13  COGS_USD             20745 non-null  float64
 14  Logistics_Cost_USD   20745 non-null  float64
 15  Net_Revenue_USD      20745 non-null  str    
 1

In [5]:
df['Geo_ID'] = df['Geo_ID'].fillna(-1)
df['SKU'] = df['SKU'].fillna('Unknown')
df['SalesPerson_ID'] = df['SalesPerson_ID'].fillna(-1)
df['CustomerType_ID'] = df['CustomerType_ID'].fillna(-1)
df['Channel_ID'] = df['Channel_ID'].fillna(-1)
df['Promotion_ID'] = df['Promotion_ID'].fillna(-1)

In [6]:
def get_pattern(date_str):
    if pd.isna(date_str):
        return None
    return re.sub(r'\d', '#', str(date_str))

In [7]:
def check_bad_values(df, col):
    temp = pd.to_numeric(df[col], errors='coerce')
    bad = df[temp.isna() & df[col].notna()][col].unique()
    print(f"Số lỗi khác nhau: {len(bad)}")
    return bad

In [8]:
df['Net_Revenue_USD'].apply(get_pattern).value_counts()

Net_Revenue_USD
###.##       11791
####.##       4631
##.##         1264
###.#         1211
$###.##        681
####.#         467
$#,###.##      279
##.#           145
###            140
$##.##          77
####            52
##               7
Name: count, dtype: int64

In [9]:
check_bad_values(df, 'Net_Revenue_USD')

Số lỗi khác nhau: 1034


<ArrowStringArray>
['$1,999.88',   '$295.85',   '$460.33',   '$156.65',   '$180.41', '$2,072.59',
    '$86.03', '$1,447.71',   '$645.44',   '$190.80',
 ...
   '$982.33',   '$761.26',   '$589.18',   '$772.73',   '$748.24',   '$165.37',
   '$829.19', '$1,815.62',   '$472.25',    '$88.84']
Length: 1034, dtype: str

In [10]:
df['Net_Revenue_USD'] = df['Net_Revenue_USD'].str.replace('$', '', regex=False)
df['Net_Revenue_USD'] = df['Net_Revenue_USD'].str.replace(',', '', regex=False)
df['Net_Revenue_USD'] = pd.to_numeric(df['Net_Revenue_USD'], errors='coerce')
df['Net_Revenue_USD'].dtype

dtype('float64')

In [11]:
df['Profit_Margin_Pct'].apply(get_pattern).value_counts()

Profit_Margin_Pct
##.##      14922
#.##        2055
##.#        1494
##.##%       777
-#.##        564
#.#          203
##           193
-##.##       156
#.##%        106
##.#%         90
-#.#          65
#             29
-##.##%       22
-##.#         19
-#.##%        19
#.#%          17
-#             7
-#.#%          5
-##.#%         1
-##            1
Name: count, dtype: int64

In [12]:
 check_bad_values(df, 'Profit_Margin_Pct')

Số lỗi khác nhau: 889


<ArrowStringArray>
[ '26.61%',  '11.11%',  '23.32%',  '19.78%',  '19.65%',   '21.9%',   '8.25%',
  '35.26%',  '26.29%',  '26.85%',
 ...
  '30.37%',  '19.93%',   '4.47%',  '14.68%',  '26.98%',  '24.27%',  '14.38%',
 '-27.64%',  '-4.01%',   '11.1%']
Length: 889, dtype: str

In [13]:
df['Profit_Margin_Pct'] = df['Profit_Margin_Pct'].str.replace('%', '', regex=False)
df['Profit_Margin_Pct'] = pd.to_numeric(df['Profit_Margin_Pct'], errors='coerce')
df['Profit_Margin_Pct'].dtype

dtype('float64')

In [14]:
df['Order_Date'].apply(get_pattern).value_counts()

Order_Date
####-##-## ##:##:##    14521
####-##-##              2828
##/##/####              1526
##-##-####              1275
Name: count, dtype: int64

In [15]:
def parse_multi_format(date_str):
    if pd.isna(date_str):
        return pd.NaT
    for fmt in ('%Y-%m-%d %H:%M:%S', '%Y-%m-%d', '%d/%m/%Y', '%m-%d-%Y'):
        try:
            return pd.to_datetime(date_str, format=fmt)
        except ValueError:
            continue
    return pd.NaT

In [16]:
df['Order_Date_Cleaned'] = df['Order_Date'].apply(parse_multi_format)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20745 entries, 0 to 20744
Data columns (total 19 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Order_ID             20745 non-null  str           
 1   Order_Date           20150 non-null  str           
 2   Geo_ID               20745 non-null  float64       
 3   SKU                  20745 non-null  str           
 4   SalesPerson_ID       20745 non-null  float64       
 5   CustomerType_ID      20745 non-null  float64       
 6   Channel_ID           20745 non-null  float64       
 7   Promotion_ID         20745 non-null  float64       
 8   Units_Sold           20745 non-null  float64       
 9   Unit_Price_USD       20745 non-null  float64       
 10  Discount_Pct         20745 non-null  float64       
 11  Gross_Sales_USD      20745 non-null  float64       
 12  Marketing_Spend_USD  20745 non-null  float64       
 13  COGS_USD             20745 non-null  float

In [17]:
df.describe()

,Geo_ID,SalesPerson_ID,CustomerType_ID,Channel_ID,Promotion_ID,Units_Sold,Unit_Price_USD,Discount_Pct,Gross_Sales_USD,Marketing_Spend_USD,COGS_USD,Logistics_Cost_USD,Net_Revenue_USD,Profit_USD,Profit_Margin_Pct,Order_Date_Cleaned
count,20745.000000,20745.000000,20745.000000,20745.000000,20745.000000,20745.000000,20745.000000,20745.000000,20745.000000,20745.000000,20745.000000,20745.000000,20745.000000,20745.000000,20745.000000,20150
mean,122.449072,17.990022,1.445698,10.335840,12.333671,209.541133,4.457012,14.054876,943.905424,86.225828,474.373175,54.433265,798.251131,182.962001,19.928385,2024-08-15 13:33:37.071960
min,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-990.000000,0.000000,0.000000,13.020000,1.810000,6.310000,1.080000,10.540000,-637.500000,-45.310000,2023-01-01 00:00:00
25%,10.000000,9.000000,1.000000,2.000000,2.000000,68.000000,3.020000,8.770000,274.960000,33.650000,140.670000,21.450000,245.590000,32.770000,13.770000,2023-10-24 00:00:00
50%,23.000000,18.000000,1.000000,2.000000,6.000000,138.000000,3.990000,12.760000,568.260000,61.610000,288.900000,39.080000,496.620000,92.310000,20.870000,2024-08-17 00:00:00
75%,36.000000,27.000000,2.000000,3.000000,6.000000,286.000000,5.450000,17.030000,1228.460000,108.420000,603.910000,69.770000,1035.360000,241.490000,27.290000,2025-06-05 00:00:00
max,9999.000000,36.000000,2.000000,999.000000,999.000000,1391.000000,16.100000,198.100000,11016.720000,1767.710000,6473.330000,628.860000,9312.430000,2723.000000,44.490000,2026-03-31 00:00:00
std,991.669764,10.602103,0.605886,88.803806,88.647643,207.783460,1.978958,13.714665,1056.287238,87.955693,542.707624,51.761017,869.870808,241.062125,10.490512,NaN


In [18]:
print("Geo_ID lạ:", df[~df['Geo_ID'].isin(region['Geo_ID'])]['Geo_ID'].unique())
print("Channel_ID lạ:", df[~df['Channel_ID'].isin(sales_channel['Channel_ID'])]['Channel_ID'].unique())
print("Promotion_ID lạ:", df[~df['Promotion_ID'].isin(promotion_type['Promotion_ID'])]['Promotion_ID'].unique())

Geo_ID lạ: [-1.000e+00  9.999e+03]
Channel_ID lạ: [ -1. 999.]
Promotion_ID lạ: [999.  -1.]


In [19]:
df.loc[df['Geo_ID'] == 9999, 'Geo_ID'] = -1
df.loc[df['Channel_ID'] == 999, 'Channel_ID'] = -1
df.loc[df['Promotion_ID'] == 999, 'Promotion_ID'] = -1

In [20]:
customer_type.info()
product_type.info()
promotion_type.info()
region.info()
sales_channel.info()
sales_person.info()

<class 'pandas.DataFrame'>
RangeIndex: 2 entries, 0 to 1
Data columns (total 2 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   CustomerType_ID  2 non-null      int64
 1   Customer_Type    2 non-null      str  
dtypes: int64(1), str(1)
memory usage: 170.0 bytes
<class 'pandas.DataFrame'>
RangeIndex: 32 entries, 0 to 31
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   SKU               32 non-null     str  
 1   Product_Category  32 non-null     str  
 2   Brand             32 non-null     str  
 3   Product_Name      32 non-null     str  
dtypes: str(4)
memory usage: 2.6 KB
<class 'pandas.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 2 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   Promotion_ID    7 non-null      int64
 1   Promotion_Type  7 non-null      str  
dtypes: int64(1), str(1)
m

In [21]:
sales_person['Sales_Person'] = sales_person['Sales_Person'].fillna('Unknown')

In [22]:
customer_type = pd.concat([customer_type, pd.DataFrame({'CustomerType_ID': [-1], 'Customer_Type': ['Unknown']})], ignore_index=True)
product_type = pd.concat([product_type, pd.DataFrame({'SKU': ['Unknown'], 'Product_Category': ['Unknown'], 'Brand': ['Unknown'], 'Product_Name': ['Unknown']})], ignore_index=True)
promotion_type = pd.concat([promotion_type, pd.DataFrame({'Promotion_ID': [-1], 'Promotion_Type': ['Unknown']})], ignore_index=True)
region = pd.concat([region, pd.DataFrame({'Geo_ID': [-1], 'Region': ['Unknown'], 'Country': ['Unknown'], 'City': ['Unknown']})], ignore_index=True)
sales_channel = pd.concat([sales_channel, pd.DataFrame({'Channel_ID': [-1], 'Sales_Channel': ['Unknown']})], ignore_index=True)
sales_person = pd.concat([sales_person, pd.DataFrame({'SalesPerson_ID': [-1], 'Sales_Person': ['Unknown']})], ignore_index=True)

In [23]:
customer_type.info()
product_type.info()
promotion_type.info()
region.info()
sales_channel.info()
sales_person.info()

<class 'pandas.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 2 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   CustomerType_ID  3 non-null      int64
 1   Customer_Type    3 non-null      str  
dtypes: int64(1), str(1)
memory usage: 193.0 bytes
<class 'pandas.DataFrame'>
RangeIndex: 33 entries, 0 to 32
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   SKU               33 non-null     str  
 1   Product_Category  33 non-null     str  
 2   Brand             33 non-null     str  
 3   Product_Name      33 non-null     str  
dtypes: str(4)
memory usage: 2.6 KB
<class 'pandas.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 2 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   Promotion_ID    8 non-null      int64
 1   Promotion_Type  8 non-null      str  
dtypes: int64(1), str(1)
m

In [24]:
dimension_tables = {'customer_type': customer_type,'product_type': product_type,'promotion_type': promotion_type,'region': region,'sales_channel': sales_channel,'sales_person': sales_person}

In [25]:
def clean_dim_df(df, key_cols=None):
    df_clean = df.copy()
    key_cols = key_cols or []
    str_cols = df_clean.select_dtypes(include=['object', 'string']).columns
    str_cols = [c for c in str_cols if c not in key_cols]
    for col in str_cols:
        df_clean[col] = df_clean[col].astype(str).str.strip().str.replace(r'\s+', ' ', regex=True).str.title()
    return df_clean.drop_duplicates()

In [26]:
cleaned_dims = {name: clean_dim_df(dim_df, key_cols=['SKU'] if name == 'product_type' else []) for name, dim_df in dimension_tables.items()}
customer_type = cleaned_dims['customer_type']
product_type = cleaned_dims['product_type']
promotion_type = cleaned_dims['promotion_type']
region = cleaned_dims['region']
sales_channel = cleaned_dims['sales_channel']
sales_person = cleaned_dims['sales_person']

In [27]:
df_merged = df.copy()
df_merged = pd.merge(df_merged, customer_type, on='CustomerType_ID', how='left')
df_merged = pd.merge(df_merged, product_type, on='SKU', how='left')
df_merged = pd.merge(df_merged, promotion_type, on='Promotion_ID', how='left')
df_merged = pd.merge(df_merged, region, on='Geo_ID', how='left')
df_merged = pd.merge(df_merged, sales_channel, on='Channel_ID', how='left')
df_merged = pd.merge(df_merged, sales_person, on='SalesPerson_ID', how='left')
print(f"Số dòng bảng gốc: {len(df)}")
print(f"Số dòng sau JOIN: {len(df_merged)}")

Số dòng bảng gốc: 20745
Số dòng sau JOIN: 20745


In [28]:
df_merged.info()

<class 'pandas.DataFrame'>
RangeIndex: 20745 entries, 0 to 20744
Data columns (total 29 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Order_ID             20745 non-null  str           
 1   Order_Date           20150 non-null  str           
 2   Geo_ID               20745 non-null  float64       
 3   SKU                  20745 non-null  str           
 4   SalesPerson_ID       20745 non-null  float64       
 5   CustomerType_ID      20745 non-null  float64       
 6   Channel_ID           20745 non-null  float64       
 7   Promotion_ID         20745 non-null  float64       
 8   Units_Sold           20745 non-null  float64       
 9   Unit_Price_USD       20745 non-null  float64       
 10  Discount_Pct         20745 non-null  float64       
 11  Gross_Sales_USD      20745 non-null  float64       
 12  Marketing_Spend_USD  20745 non-null  float64       
 13  COGS_USD             20745 non-null  float

In [29]:
df_merged.describe()

,Geo_ID,SalesPerson_ID,CustomerType_ID,Channel_ID,Promotion_ID,Units_Sold,Unit_Price_USD,Discount_Pct,Gross_Sales_USD,Marketing_Spend_USD,COGS_USD,Logistics_Cost_USD,Net_Revenue_USD,Profit_USD,Profit_Margin_Pct,Order_Date_Cleaned
count,20745.000000,20745.000000,20745.000000,20745.000000,20745.000000,20745.000000,20745.000000,20745.000000,20745.000000,20745.000000,20745.000000,20745.000000,20745.000000,20745.000000,20745.000000,20150
mean,22.665992,17.990022,1.445698,2.333912,4.331743,209.541133,4.457012,14.054876,943.905424,86.225828,474.373175,54.433265,798.251131,182.962001,19.928385,2024-08-15 13:33:37.071960
min,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-990.000000,0.000000,0.000000,13.020000,1.810000,6.310000,1.080000,10.540000,-637.500000,-45.310000,2023-01-01 00:00:00
25%,10.000000,9.000000,1.000000,2.000000,2.000000,68.000000,3.020000,8.770000,274.960000,33.650000,140.670000,21.450000,245.590000,32.770000,13.770000,2023-10-24 00:00:00
50%,23.000000,18.000000,1.000000,2.000000,6.000000,138.000000,3.990000,12.760000,568.260000,61.610000,288.900000,39.080000,496.620000,92.310000,20.870000,2024-08-17 00:00:00
75%,35.000000,27.000000,2.000000,3.000000,6.000000,286.000000,5.450000,17.030000,1228.460000,108.420000,603.910000,69.770000,1035.360000,241.490000,27.290000,2025-06-05 00:00:00
max,48.000000,36.000000,2.000000,4.000000,7.000000,1391.000000,16.100000,198.100000,11016.720000,1767.710000,6473.330000,628.860000,9312.430000,2723.000000,44.490000,2026-03-31 00:00:00
std,14.502315,10.602103,0.605886,1.092696,2.336073,207.783460,1.978958,13.714665,1056.287238,87.955693,542.707624,51.761017,869.870808,241.062125,10.490512,NaN


In [30]:
print("Số dòng Units_Sold < 0:", (df_merged['Units_Sold'] < 0).sum())
print("Số dòng Unit_Price_USD == 0:", (df_merged['Unit_Price_USD'] == 0).sum())
print("Số dòng Discount_Pct > 100:", (df_merged['Discount_Pct'] > 100).sum())

Số dòng Units_Sold < 0: 207
Số dòng Unit_Price_USD == 0: 104
Số dòng Discount_Pct > 100: 166


In [31]:
df_merged['Transaction_Type'] = df_merged['Units_Sold'].apply(lambda x: 'Return' if x < 0 else 'Sale')
df_merged.loc[df_merged['Discount_Pct'] > 100, 'Discount_Pct'] = 100
df_merged = df_merged[(df_merged['Unit_Price_USD'] > 0).copy()

In [32]:
import numpy as np
df_merged['Gross_Sales_USD'] = df_merged['Units_Sold'] * df_merged['Unit_Price_USD']
df_merged['Net_Revenue_USD'] = df_merged['Gross_Sales_USD'] * (1 - df_merged['Discount_Pct'] / 100)
df_merged['Profit_USD'] = df_merged['Net_Revenue_USD'] - df_merged['COGS_USD'] - df_merged['Marketing_Spend_USD'] - df_merged['Logistics_Cost_USD']
df_merged['Profit_Margin_Pct'] = np.where(df_merged['Net_Revenue_USD'] != 0, (df_merged['Profit_USD'] / df_merged['Net_Revenue_USD']) * 100, 0)

In [33]:
df_merged.describe()

,Geo_ID,SalesPerson_ID,CustomerType_ID,Channel_ID,Promotion_ID,Units_Sold,Unit_Price_USD,Discount_Pct,Gross_Sales_USD,Marketing_Spend_USD,COGS_USD,Logistics_Cost_USD,Net_Revenue_USD,Profit_USD,Profit_Margin_Pct,Order_Date_Cleaned
count,20434.000000,20434.000000,20434.000000,20434.000000,20434.000000,20434.000000,20434.000000,20434.000000,20434.000000,20434.000000,20434.000000,20434.000000,20434.000000,20434.000000,20434.000000,19850
mean,22.668053,17.988353,1.445972,2.334051,4.330528,213.946801,4.480067,13.655834,944.981128,86.311707,474.949789,54.468186,792.996258,177.266576,19.673643,2024-08-15 03:58:48.906801
min,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,5.000000,1.330000,0.000000,13.020000,1.810000,6.310000,1.080000,0.000000,-3478.290000,-97.727618,2023-01-01 00:00:00
25%,10.000000,9.000000,1.000000,2.000000,2.000000,70.000000,3.030000,8.780000,275.140000,33.690000,140.802500,21.480000,241.186190,31.127313,13.418528,2023-10-24 00:00:00
50%,23.000000,18.000000,1.000000,2.000000,6.000000,140.000000,4.010000,12.760000,568.590000,61.715000,289.390000,39.110000,492.034848,90.680420,20.767879,2024-08-15 00:00:00
75%,35.000000,27.000000,2.000000,3.000000,6.000000,289.000000,5.457500,17.030000,1229.625000,108.455000,604.142500,69.755000,1029.302391,240.068912,27.361007,2025-06-05 00:00:00
max,48.000000,36.000000,2.000000,4.000000,7.000000,1391.000000,16.100000,100.000000,11016.720000,1767.710000,6473.330000,628.860000,9312.433416,2723.004172,58.329037,2026-03-31 00:00:00
std,14.505914,10.604951,0.605480,1.092810,2.336281,203.379186,1.957732,9.647017,1058.076674,88.105626,543.781001,51.824441,871.424911,259.818033,11.085002,NaN


In [34]:
df_merged.to_sql(name='Fact_Sales_Merged', con=conn, if_exists='replace', index=False)

20434

In [35]:
conn.close()